## Decorators 

A decorator is a function that wraps another function to add extra behaviour before or after it runs - such as timing execution, checking user permission, or retrying failed netowrk call without chainging the original function's internal code.

## Two Python rule behind Decorators

Decorators work in Python becuase of two core feautes

- Functions are objects; We can assign a function to a variable, pass it into another function as an argument, and we can return a function from inside of another function

- Closures: The function inside the another function can remembers the variables of even the outer function that created it, even after the outer function finishes running.

### Building our First Decorator (Execution Timer)

**Example 1: Timing a warehouse inverntory reconciliation job**

Suppose we want to measure how many mullieseconds a slow batch calculation takes without cluttering the calculation logic itself with time.perf_counter() calls

In [7]:
import functools
import time
from typing import Any, Callable

def measures_time(func: Callable[..., Any])-> Callable[...,Any]:
    @functools.wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        start_time = time.perf_counter()
        result = func(*args, *kwargs) # Runs the original function
        elapsed_ms = (time.perf_counter() - start_time) * 1000
        print(f"[TIMER] '{func.__name__}' finished in {elapsed_ms:.2f} ms")
        return result
    return wrapper


@measures_time
def  reconcile_warehouse_stock(sku_count: int) -> int:
    """ Simulates scanning and summing stock items"""
    total = sum(range(sku_count))
    return total

# Calling the decorated function actually calls wrapped(500_000)
final_count = reconcile_warehouse_stock(500_000_000)
print("Total units reconciled:", final_count)
print("Preserved function name:", reconcile_warehouse_stock.__name__)

[TIMER] 'reconcile_warehouse_stock' finished in 4976.72 ms
Total units reconciled: 124999999750000000
Preserved function name: reconcile_warehouse_stock


### How this code works:

- @measure_ti e placed above def reconcile_warehouse_stock is purely shorthand of writing
    - reconcile_warehouse_stock = mesure_time(reconciel_warehouse_stock)

- measure_time recieves reconcile_warehouse_stock as func, defines wrapper,and returns wrapper. From that point onward, the name reconcile_warehouse_stock actually points to wrapper

- *args and **kwargs inside the wrapper allow the decorator to accept any positional or keyword arguments forward them untouched into func(*args, **kwargs)
- @functools.wraps(func) copies the original function's __name__ and docstring onto wrapper = sp bebugging tools and frameworks don't mistankely see 'wrapper" instead of "reconcile_warehouse_stock"

## Real-worls Guardrail Decorator (Access Control)

Decorators are widely used to intercept function calls and block execution if a condition is not met.

**Example 2 : Hospital Pharmacy Controlled- Substance Dispenser**

Only staff with the "pharmacist" role should be allowed to approave restricted medication orders

In [10]:
import functools
from typing import Callable, Any

def require_pharmacist(func: Callable[..., Any]) -> Callable[..., Any]:
    @functools.wraps(func)
    def wrapper(staff_role: str, *args: Any, **kwargs: Any) -> Any:
    
        if staff_role.lower() != "pharmacist":
            raise PermissionError(
                f"Access denied: role '{staff_role} cannot execute '{func.__name__}"

            )
        return func(staff_role, *args, *kwargs)
    return wrapper


@require_pharmacist
def dispense_controlled_drug(staff_role: str, patient_id: str, drug_name: str) -> str:
    return f"Approved by {staff_role}: Dispensed {drug_name} for {patient_id}"

# 1. Authorized call succeeds

print(dispense_controlled_drug("pharmacist","PAT-882", "Morphine 5mg"))

# 2. Unauthorized call is intercepted before the function body ever runs:

try:
    print(dispense_controlled_drug("receptionist", "PAT-882", "Morphine 5mg"))
except PermissionError as err:
    print(err)



Approved by pharmacist: Dispensed Morphine 5mg for PAT-882
Access denied: role 'receptionist cannot execute 'dispense_controlled_drug


### How this code works:

when dispense_controlled_drug("receptionis",...) is called , werapper inspects the first argument staff_role before calling func.


because staff_role is not correct, werapper raise the error immediately preventing the dispence function from ever executing


when pharmacist is passed, wrappr fowards all arguments to the original function and returs its string output

### Example 3 - Automatic Retyr for a Flaky SMS Gateway



In [11]:
import functools 

from typing import Any, Callable


def retry(max_attempts: int = 3) -> Callable[[Callable[..., Any]], Callable[...,Any]] :
    def decorator(func: Callable[..., Any]) -> Callable[...,Any]:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> None:
            last_error: Exception | None = None

            for attempt in range(1, max_attempts + 1 ):
                try:
                    return func(*args, **kwargs)
                except RuntimeError as err:
                    last_error = err
                    print(f"[RETRY {attempt}/{max_attempts}] '{func.__name__}' fialed: {err}")
            raise RuntimeError(f"All {max_attempts} attempts failed.") from last_error
        return wrapper
    return decorator

atttempt_counter = 0

@retry(max_attempts=3)
def send_otp_sms(phone:str) -> str:
    global atttempt_counter
    atttempt_counter += 1
    if atttempt_counter < 3:
        raise RuntimeError("Telecom gateway timeout")
    return f"OTP delivered to {phone}"


print(send_otp_sms("+91-98534523245"))


[RETRY 1/3] 'send_otp_sms' fialed: Telecom gateway timeout
[RETRY 2/3] 'send_otp_sms' fialed: Telecom gateway timeout
OTP delivered to +91-98534523245


### HOw this code works:

- Python evaluates @retry(max_attempts=3) first, which returns decorator with max_attempts = 3 locked in its closure.

- Python then passes send_otp_sms into decorator, return wrapper.

- When send_otp_sms() runs,wrapper loops upto 3 times: attempts 1 and 2 catch RUntimeError and retry, and attempt 3 succeeds and returns the final output